In [1]:
# ═══════════════════════════════════════
# TRY 2 — STANDARD SESSION OPENER
# ═══════════════════════════════════════

import json, os, random, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

from sklearn.metrics import precision_score, recall_score, f1_score

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

BASE = "/kaggle/input/datasets/aadityaupadhyay1353"

CFG_PATH = f"{BASE}/cps-config/config.json"

with open(CFG_PATH) as f:
    CFG = json.load(f)

OUT = "/kaggle/working/"
os.makedirs(OUT, exist_ok=True)

print("TRY 2 aggregation notebook ready")

TRY 2 aggregation notebook ready


In [2]:
preds = pd.read_csv(
    f"{BASE}/v2-node-predictions/v2_node_predictions_val.csv"
)

weights_df = pd.read_csv(
    f"{BASE}/v2-node-predictions/v2_node_f1_weights.csv"
)

node_weights = dict(zip(weights_df["node_id"], weights_df["weight"]))

print("Predictions shape:", preds.shape)
print("Nodes:", preds["node_id"].unique())
print("Weights:", node_weights)

Predictions shape: (16050, 6)
Nodes: ['node_A' 'node_B' 'node_C' 'node_D' 'node_E']
Weights: {'node_A': 0.2090113813289468, 'node_B': 0.2051674692965324, 'node_C': 0.1917411546813628, 'node_D': 0.2128120489491847, 'node_E': 0.181267945743973}


In [3]:
TAU = 0.5

N_NODES = preds["node_id"].nunique()

print("Total nodes:", N_NODES)

Total nodes: 5


In [4]:
results = []

for eid in preds["engine_id"].unique():

    eng_df = preds[preds["engine_id"] == eid].sort_values("cycle")

    for t in eng_df["cycle"].unique():

        t_df = eng_df[eng_df["cycle"] == t]

        true_label = t_df["true_label"].iloc[0]

        probs = t_df["prob"].values

        flags = (probs > TAU).astype(int)

        # ----------------------
        # Majority Vote
        # ----------------------

        majority_pred = int(flags.sum() > N_NODES / 2)

        # ----------------------
        # Weighted Vote
        # ----------------------

        weights = np.array([
            node_weights[nid]
            for nid in t_df["node_id"].values
        ])

        weights = weights / weights.sum()

        weighted_score = (weights * flags).sum()

        weighted_pred = int(weighted_score > 0.5)

        # ----------------------
        # Risk-Based Aggregation
        # ----------------------

        fault_fraction = flags.sum() / N_NODES

        if fault_fraction < 0.2:

            risk_level = "SAFE"
            risk_pred = 0

        elif fault_fraction < 0.6:

            risk_level = "WARNING"
            risk_pred = 1

        else:

            risk_level = "CRITICAL"
            risk_pred = 1

        mean_prob = probs.mean()

        results.append({

            "engine_id": eid,
            "cycle": t,
            "true_label": true_label,

            "majority_pred": majority_pred,
            "weighted_pred": weighted_pred,

            "risk_pred": risk_pred,
            "risk_level": risk_level,

            "mean_prob": mean_prob,
            "fault_fraction": fault_fraction
        })

In [5]:
agg_df = pd.DataFrame(results)

print("Aggregation dataset shape:", agg_df.shape)

agg_df.head()

Aggregation dataset shape: (3210, 9)


,engine_id,cycle,true_label,majority_pred,weighted_pred,risk_pred,risk_level,mean_prob,fault_fraction
0,71,1,0,0,0,0,SAFE,0.000329,0.0
1,71,2,0,0,0,0,SAFE,0.000349,0.0
2,71,3,0,0,0,0,SAFE,0.000491,0.0
3,71,4,0,0,0,0,SAFE,0.000599,0.0
4,71,5,0,0,0,0,SAFE,0.000365,0.0


In [6]:
y_true = agg_df["true_label"]

comparison = []

strategies = {

    "Majority": "majority_pred",
    "Weighted-F1": "weighted_pred",
    "Risk-Based": "risk_pred"
}

for name, col in strategies.items():

    precision = precision_score(y_true, agg_df[col])
    recall = recall_score(y_true, agg_df[col])
    f1 = f1_score(y_true, agg_df[col])

    comparison.append({

        "strategy": name,
        "precision": precision,
        "recall": recall,
        "f1": f1
    })

    print(name, "F1:", round(f1,4))

Majority F1: 0.8554
Weighted-F1 F1: 0.8554
Risk-Based F1: 0.7415


In [7]:
comparison_df = pd.DataFrame(comparison)

comparison_df.to_csv(
    f"{OUT}v2_aggregation_comparison.csv",
    index=False
)

comparison_df

,strategy,precision,recall,f1
0,Majority,0.80618,0.911111,0.855440
1,Weighted-F1,0.80618,0.911111,0.855440
2,Risk-Based,0.60198,0.965079,0.741463


In [8]:
agg_df.to_csv(
    f"{OUT}v2_aggregation_results_full.csv",
    index=False
)

print("Aggregation results saved")

Aggregation results saved


In [9]:
print("Risk level distribution")

print(agg_df["risk_level"].value_counts())

Risk level distribution
risk_level
SAFE        2705
CRITICAL     356
WARNING      149
Name: count, dtype: int64


In [10]:
!ls -lh /kaggle/working/

total 156K
---------- 1 root root  20K Apr 12 13:15 __notebook__.ipynb
-rw-r--r-- 1 root root  231 Apr 12 13:15 v2_aggregation_comparison.csv
-rw-r--r-- 1 root root 131K Apr 12 13:15 v2_aggregation_results_full.csv
